In [8]:
from mmseg.datasets import ADE20KDataset
from mmcv.utils import build_from_cfg
import torchvision.transforms as transforms

# 自定义图像增强pipeline的transform
def image_transform(image, resolution=256, normalize=True):
    image = transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BICUBIC)(image)
    image = transforms.CenterCrop((resolution, resolution))(image)
    image = transforms.ToTensor()(image)
    if normalize:
        image = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5], inplace=True)(image)
    return image

# 注册自定义数据集类
class CustomADE20KDataset(ADE20KDataset):
    """自定义ADE20K数据集，添加自定义的数据增强方法"""

    def __init__(self, **kwargs):
        super(CustomADE20KDataset, self).__init__(**kwargs)

    # 重写pipeline以适应自定义的图像处理逻辑
    def pre_pipeline(self, results):
        super(CustomADE20KDataset, self).pre_pipeline(results)
        
        # 自定义图像增强操作，替换pipeline中的图像处理步骤
        img = results['img']
        img = image_transform(img, resolution=256, normalize=True)
        results['img'] = img

# 使用自定义的dataset类
dataset = CustomADE20KDataset(
    ann_dir='path_to_annotations',  # 标注文件路径
    img_dir='path_to_images',       # 图像文件路径
    pipeline=[
        dict(type='LoadImageFromFile'),   # 加载图像
        dict(type='LoadAnnotations'),     # 加载标注文件
        dict(type='CustomTransform'),     # 自定义图像增强操作，调用自定义的Transform
    ],
    split='train',  # 训练集、验证集或测试集
    data_root='path_to_ade20k_dataset'
)


In [12]:
# Example usage:
training_dataset = CustomADE20KDataset(data_root='/data/junzhe/datasets/ade/ADEChallengeData2016')
# print(training_dataset.get_data_info(0))
# print(training_dataset.metainfo)
print(training_dataset[0])


{'img_path': '/data/junzhe/datasets/ade/ADEChallengeData2016/images/training/ADE_train_00000001.jpg', 'seg_map_path': '/data/junzhe/datasets/ade/ADEChallengeData2016/annotations/training/ADE_train_00000001.png', 'label_map': None, 'reduce_zero_label': True, 'seg_fields': [], 'sample_idx': 0, 'image': tensor([[[ 0.4588,  0.4588,  0.4667,  ...,  0.4118,  0.4118,  0.4196],
         [ 0.4588,  0.4588,  0.4667,  ...,  0.4118,  0.4118,  0.4196],
         [ 0.4667,  0.4588,  0.4667,  ...,  0.4118,  0.4118,  0.4196],
         ...,
         [-0.0980, -0.1137, -0.1216,  ..., -0.6314, -0.6000, -0.5922],
         [-0.1059, -0.1059, -0.1216,  ..., -0.6157, -0.6078, -0.5922],
         [-0.0980, -0.1059, -0.1216,  ..., -0.5843, -0.5765, -0.5608]],

        [[ 0.4902,  0.4902,  0.4980,  ...,  0.4824,  0.4824,  0.4902],
         [ 0.4902,  0.4902,  0.4980,  ...,  0.4824,  0.4824,  0.4902],
         [ 0.4980,  0.4902,  0.4980,  ...,  0.4824,  0.4824,  0.4902],
         ...,
         [-0.2784, -0.2784, -